# A Transformer for Fundamental Analysis
In this project, we build a transformer from scratch and train it on a huge dataset of financial news. Then we use the transformer for the fundamental analysis of stocks, generating sentiment signals to enhance stock-price forecasting.

## Stage 1: The Transformer (building and training)

### Stage 1.1: The Transformer Architecture
The first architectural choice to make is whether to build a decoder-only, encoder-only, or encoder-decoder transformer. We know that a decoder-only transformer is typically used for sequence generation/continuation from a single token, an encoder-only transformer is typically used for representation learning, and an encoder-decoder is typically used for sequence-to-sequence translation. For our fundamental analysis, we want to use the model for sentiment analysis, for which representation learning is the most useful, hence we choose to use an encoder-only transformer.

In [ ]:
import torch
import torch.nn as nn
import math
import string
import random
# In this file, we will implement the architecture of a simple transformer encoder model.

# TODO: implement the transformer using the transformer layer
class Transformer(nn.Module):
    def __init__(self, d_model, d_internal, num_layers):
        super().__init__()
        layers = [TransformerLayer(d_model, d_internal) for i in range(num_layers)]
        self.transformer = nn.Sequential(*layers)
        self.linear_classifier = nn.Sequential(nn.Linear(d_model, d_model))

    def forward(self, x):
        if not torch.is_tensor(x):
            raise TypeError(f"Input is not Torch tensor: {type(x).__name__}")
        return self.transformer(x)
    
    def predict():
        pass
        # 1. Mask
        # 2. Foward
        # 3. return prediction


# TODO: add a causal flag with causal implementation
# TODO: add positional encodings
class TransformerLayer(nn.Module): 

    # 1. initialize learnable parameters
    #   d_model: size of each vector embedding (for each input and each output)
    #   d_internal: size of query vectors, key vectors, and FFN hidden layer
    def __init__(self, d_model, d_internal):
        super().__init__()

        # 1- define self-attention
        self.q = nn.Linear(d_model, d_internal)
        self.k = nn.Linear(d_model, d_internal)
        self.v = nn.Linear(d_model, d_model)
        
        self.scores = lambda x: self.q(x) @ self.k(x).transpose(-2, -1) / math.sqrt(d_internal)
        self.attn = lambda x: torch.softmax(self.scores(x), dim=-1) @ self.v(x)

        # 2- define ffn (MLP with L layers, each layer with m_l units)
        self.fc1 = nn.Linear(d_model, d_internal)
        self.activation = nn.ReLU()
        self.fc2 = nn.Linear(d_internal, d_model)

        self.ffn = lambda z: self.fc2(self.activation(self.fc1(z)))

        # 3- define layernorm
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    # 2. define the computation graph
    def forward(self, x):
        z = self.ln1(x + self.attn(x))
        y = self.ln2(z + self.ffn(z))
        return y


class transformer_language_model():
    def __init__():
        pass
    def train():
        pass
    def predict():
        pass


# TODO: TEST & DEBUG
# training & evaluation
def train_mlm(input_string):
    # 1. Tokenize (turn input_string into a sequence of token x)
    # lowercase, 
    # split by whitespace, 
    # then in each word, drop trailing punctuations in string.punctuation except "$%+-"
    tokens = input_string.lower().split()
    for k in range(len(tokens)):
        token = tokens[k]
        i = len(token) - 1
        while token[i] in string.punctuation and token[i] not in "$%+-": 
            i -= 1
        tokens[k] = token[:i+1]
    types = set(tokens)
    vocab_size = len(list(types))
    one_hots = dict()
    type_id = 0
    for type in types:
        one_hot = torch.zeros(vocab_size)
        one_hot[type_id] = 1
        one_hots.update({type : one_hot})
        type_id += 1
    
    chunk_size = 5
    training_examples = []
    i = 0
    n = len(tokens)
    num_chunks = n // 20
    for i in range(num_chunks):
        training_examples.append(tokens[i*20:(i+1)*20])
    if num_chunks * 20 < n: 
        training_examples.append(tokens[(num_chunks)*20:])
    d_model = vocab_size
    d_internal = 16
    num_layers = 2
    model = Transformer(d_model=d_model, d_internal=d_internal, num_layers=num_layers)
    # 2. Forward
    print(one_hots)
    print(tokens)
    x = torch.stack(list(map(lambda token: one_hots[token], tokens))) # apply one-hot encodings
    y = model.forward(x)
    # 3. MLM loss
    # 3.1 Set up learning rate, loss function, and optimizer
    lr = 1e-4
    loss_fn = nn.CrossEntropyLoss()   # CrossEntropyLoss = NLLLoss(Softmax(logits))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    p = 15
    mask_indices = random.sample(k = int(len(tokens) * (p/100)), population = range(len(tokens))) # randomly mask p% of tokens?
    preds = torch.stack([model.linear_classifier(y[i]) for i in mask_indices])
    targets = torch.stack([x[i] for i in mask_indices])
    loss = loss_fn(preds, targets)
    # 4. Backward
    loss.backward()
    # 5. Optimize & Save
    optimizer.step()


model =  Transformer(2, 2, 2)
print(train_mlm("Ahmed is a great math student! \s"))
sentence_with_mask = "Ahmed is a [MASK] math student! \s"

### Stage 1.2: Training on FNSPID
FNSPID is the largest publicly available dataset of financial news we could find, with 15.7 million financial news records for 4,775 S&P500 companies from 1999 to 2023.

## Stage 2: Stock Fundamental Analysis (an end-to-end pipeline)

### Stage 2.1: Stock preprocessing

### Stage 2.2: Stock Time-Series Modeling

### Stage 2.3: Signal Generation (sentiment + narrative stress)

### Stage 2.4: Hybrid Forecast (from past prices + signals)